## 🎯 Learning Objectives
* Understand the fundamental concept of conversable agents in AutoGen.
* Differentiate between AssistantAgent and UserProxyAgent roles and functionalities.
* Learn how to instantiate and initiate a basic conversation between AutoGen agents.
* Identify typical use cases and performance considerations for these core agent types.


## Conversable Agents: The Foundation of AutoGen's Collaborative AI

AutoGen's power lies in its ability to orchestrate multiple AI agents that can communicate and collaborate to solve complex tasks. At the heart of this system are **conversable agents**, which are essentially entities capable of sending and receiving messages, processing them, and responding accordingly. Think of them as participants in a highly intelligent, automated team meeting.

Every agent in AutoGen, regardless of its specific role, inherits from the `ConversableAgent` class. This base class provides the fundamental capabilities for communication, such as sending messages, receiving messages, and managing conversation history. It's like giving every team member a common language and a communication channel.

Within this framework, two primary agent types form the backbone of most AutoGen applications:

1.  **`AssistantAgent`**: This agent is your AI expert. It's designed to leverage Large Language Models (LLMs) to perform tasks, generate code, answer questions, and solve problems. You can think of the `AssistantAgent` as a highly skilled specialist (e.g., a software engineer, a data scientist, a content creator) who takes instructions, processes information, and proposes solutions. It doesn't directly interact with the human user or execute code by default; its primary role is intellectual work.

2.  **`UserProxyAgent`**: This agent acts as the bridge between the human user and other AI agents, and also as an executor of tasks. It's like a project manager or a human-in-the-loop interface. The `UserProxyAgent` can:
    *   **Initiate conversations**: Start a task by sending a message to an `AssistantAgent`.
    *   **Receive human input**: Prompt the user for feedback, clarification, or approval.
    *   **Execute code**: If an `AssistantAgent` proposes code, the `UserProxyAgent` can execute it in a local environment (e.g., a Docker container or a local Python interpreter) and report the results back to the `AssistantAgent`.
    *   **Terminate conversations**: Decide when a task is complete or when to stop the interaction.

### Analogy: The Software Development Team

Imagine a software development team working on a new feature:

*   **You (the human user)**: The product owner who defines the requirements.
*   **`UserProxyAgent`**: The Project Manager. They take your requirements, break them down, assign tasks to the engineers, review their code, run tests, and report back to you. If an engineer needs more information, the Project Manager asks you. If the engineer writes code, the Project Manager runs it to see if it works.
*   **`AssistantAgent`**: The Software Engineer. They receive tasks from the Project Manager, write code, debug issues, and propose solutions. They don't directly talk to the Product Owner (you) or run the code themselves; they hand their work back to the Project Manager.

This division of labor allows for robust, multi-step problem-solving where the `AssistantAgent` focuses on cognitive tasks, and the `UserProxyAgent` handles interaction, execution, and human oversight.


In [ ]:
import autogen
import os

# --- Configuration for LLM --- 
# As of 2026, AutoGen supports various LLM providers. 
# We'll use OpenAI's API for this example. Ensure your API key is set.
# You can also configure local LLMs or other cloud providers.

# It's best practice to load API keys from environment variables.
# For demonstration, we'll assume OPENAI_API_KEY is set in your environment.
# Example: os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Define the LLM configuration list. AutoGen can use multiple models.
config_list_openai = [
    {
        "model": "gpt-4o-2024-05-13", # Using a recent, powerful model as of 2026
        "api_key": os.environ.get("OPENAI_API_KEY")
    }
]

# --- Instantiate Agents --- 

# 1. AssistantAgent: The AI expert
# This agent will generate ideas, write code, and solve problems.
# The system_message helps define its persona and capabilities.
assistant = autogen.AssistantAgent(
    name="CoderAssistant",
    llm_config={
        "config_list": config_list_openai,
        "temperature": 0.7 # Controls creativity; lower for more deterministic output
    },
    system_message="You are a helpful AI assistant specialized in Python programming. You can write, debug, and explain Python code. Provide clear and concise solutions."
)

# 2. UserProxyAgent: The human proxy and code executor
# This agent acts on behalf of the human user and can execute code.
# human_input_mode: 
#   - "ALWAYS": Always ask for human input before replying.
#   - "NEVER": Never ask for human input (fully autonomous).
#   - "TERMINATE": Ask for human input when a 'TERMINATE' message is received.
# code_execution_config: 
#   - "policy": "auto" (default), "docker", "local"
#   - "work_dir": Directory where code will be executed.
user_proxy = autogen.UserProxyAgent(
    name="UserProxy",
    human_input_mode="NEVER", # For this demo, we want autonomous execution
    max_consecutive_auto_reply=10, # Max turns before stopping if no human input
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        "work_dir": "coding", # Code will be executed in a 'coding' subdirectory
        "use_docker": False # Set to True if you have Docker installed for isolated execution
    }
)

# --- Initiate a Conversation --- 

# The UserProxyAgent initiates the chat with the AssistantAgent.
# The message defines the initial task.
print("\n--- Starting Conversation ---\n")
user_proxy.initiate_chat(
    assistant,
    message="Write a Python function that calculates the nth Fibonacci number using recursion. Include docstrings and type hints. Then, call the function to find the 10th Fibonacci number and print the result. Finally, print 'TERMINATE' to end the conversation."
)
print("\n--- Conversation Ended ---\n")

# You can inspect the chat history if needed
# print(user_proxy.chat_messages[assistant])


### Interpreting the Code Output and Use Cases

When you run the provided code, you'll observe a dynamic conversation unfold in your console. Here's what's happening:

1.  **`UserProxy` initiates**: The `UserProxy` sends the initial task to the `CoderAssistant`.
2.  **`CoderAssistant` responds**: The `CoderAssistant`, powered by the LLM, processes the request. It understands the need for a recursive Fibonacci function, docstrings, type hints, and a test call. It then generates the Python code.
3.  **`UserProxy` executes**: The `UserProxy` receives the code from the `CoderAssistant`. Because `code_execution_config` is set, it automatically attempts to execute this code in the specified `work_dir` (a `coding` subdirectory will be created if it doesn't exist). It captures the standard output and any errors.
4.  **`UserProxy` reports back**: The `UserProxy` sends the execution results (e.g., the printed Fibonacci number) back to the `CoderAssistant`.
5.  **`CoderAssistant` acknowledges/continues**: The `CoderAssistant` receives the execution result. In this specific example, the prompt included `print 'TERMINATE'`, which the `CoderAssistant` will eventually output. The `UserProxy`'s `is_termination_msg` lambda function detects this and ends the conversation.

This interaction demonstrates a fundamental pattern in AutoGen: **plan, execute, verify, iterate**. The `AssistantAgent` plans and generates, while the `UserProxyAgent` executes and verifies, providing crucial feedback that allows the `AssistantAgent` to refine its approach if necessary.

#### Performance Trade-offs:

*   **LLM Latency**: Each turn in the conversation involves an API call to the LLM (e.g., OpenAI). This introduces latency, especially for complex tasks requiring multiple turns. Optimizing prompts and agent roles can reduce unnecessary calls.
*   **Token Usage**: More turns and longer messages mean higher token usage, directly impacting cost. Efficient communication and clear termination conditions are vital.
*   **Code Execution Overhead**: While local execution is fast, if `use_docker` is `True`, there's a slight overhead for container management. For complex environments or security, Docker is highly recommended.

#### Typical Use Cases:

This basic setup of `AssistantAgent` and `UserProxyAgent` is incredibly versatile and forms the basis for many advanced applications:

*   **Automated Code Generation and Debugging**: As seen in the example, generating Python scripts, SQL queries, or configuration files, and then automatically testing them.
*   **Data Analysis and Visualization**: An `AssistantAgent` can propose data analysis steps and code, which the `UserProxyAgent` executes, returning results for further analysis.
*   **Content Creation**: Generating articles, marketing copy, or creative writing, with the `UserProxyAgent` acting as an editor or reviewer.
*   **Task Automation**: Automating repetitive tasks by having the `AssistantAgent` define the steps and the `UserProxyAgent` execute them.
*   **Interactive Problem Solving**: When `human_input_mode` is set to `ALWAYS`, the `UserProxyAgent` can prompt the human for input at critical junctures, allowing for guided problem-solving.


### Resources

*   **AutoGen Official Documentation**: The primary source for all AutoGen features, examples, and API references.
    *   [AutoGen GitHub Repository](https://github.com/microsoft/autogen)
    *   [AutoGen Documentation](https://microsoft.github.io/autogen/)
*   **OpenAI API Documentation**: For understanding the underlying LLM capabilities and parameters.
    *   [OpenAI API Reference](https://platform.openai.com/docs/api-reference)
*   **Python `os` module**: For managing environment variables, crucial for securely handling API keys.
    *   [Python `os` module documentation](https://docs.python.org/3/library/os.html)
